In [1]:
import openai
print(openai.__version__)
print(openai.__file__)

1.30.1
C:\Users\RAZER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openai\__init__.py


In [2]:
# !pip install -r ../requirements.txt
# !pip install onnxruntime==1.18.0
# !pip install openai==1.30.1


In [3]:
import pydantic
print(pydantic.__file__)
print(pydantic.__version__)

C:\Users\RAZER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pydantic\__init__.py
2.7.4


In [4]:
import sys
import langchain
import chromadb
import openai
import os
from dotenv import load_dotenv

In [5]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


In [6]:
# Load raw documents from data/1_source/
loader = DirectoryLoader(
    "../data/1_source/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 2 documents


In [7]:
# Inspect a document
print(docs[0].page_content[:500])   # first 500 chars
print("---")
print(docs[0].metadata)             # source path, etc.

The buff-tip (Phalera bucephala) is a moth of the family Notodontidae. It is found throughout Europe and in Asia to eastern Siberia.[1] The species was first described by Carl Linnaeus in his 1758 10th edition of Systema Naturae.
Description

The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings 
---
{'source': '..\\data\\1_source\\wikipedia_article_1.txt'}


In [8]:
# Chunk the documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

Total chunks: 102
Avg chunk length: 336 chars


In [9]:
# Inspect a few chunks to sanity check
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)


--- Chunk 0 ---
The buff-tip (Phalera bucephala) is a moth of the family Notodontidae. It is found throughout Europe and in Asia to eastern Siberia.[1] The species was first described by Carl Linnaeus in his 1758 10th edition of Systema Naturae.
Description

--- Chunk 1 ---
The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings are creamy white. Seitz - Head, collar and centre of thorax brownish yellow, patagia greyish white with a black-brown double basal edge, on the transverse crest 2 black-brown transverse lines, hind margin greyish white. Abdomen yellowish grey

--- Chunk 2 ---
hind margin greyish white. Abdomen yellowish grey to yellowish brown. Forewing greyish brown, broadly white at the base and along the hind margin, with prediscal dark brown and black double band; at the apex

In [10]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [11]:
use_OpenAI = True


In [12]:
# Generate embeddings + store in ChromaDB
if use_OpenAI:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # cheapest + good

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="../chroma_DB/"
    )

    print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB ✅")

ValidationError: 1 validation error for OpenAIEmbeddings
__root__
  Client.__init__() got an unexpected keyword argument 'proxies' (type=type_error)